# ADS-509 Assignment 5.1
**Zadafiya Bhrami**
## Finetuning LLMs


## Imports and Downloads

In [1]:
try:
    import datasets, transformers, evaluate, torch  # type: ignore
except Exception:
    %pip install -q datasets transformers evaluate accelerate sentencepiece

import os, random, numpy as np, warnings
import torch
from sklearn.metrics import confusion_matrix
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer
warnings.filterwarnings('ignore')



[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## Load Dataset and Model

For this assignment, you will be comparing performance on a common language model benchmarking task: predicting the sentiment for the Stanford Sentiment Treebank ([SST-2](https://nlp.stanford.edu/sentiment/index.html)). We will use the same model, [Flan-T5-Small](https://huggingface.co/google/flan-t5-small), across all of our training methods so that the results are directly comparable.

**TODO**:

- Use your preferred method to select a sample of 2000 sentences from the train dataset.

**Q**: After reading a little bit about the Sentiment Treebank project at the link above, and recognizing that the paper was written in 2013, what method do we now use to provide the same kind of benefit that they intended with their tree-based sentiment representations?

**A**: The SST paper used hand-crafted constituency parse trees to capture compositional meaning -- for example, understanding that 'not bad' is positive even though 'bad' is negative. Today, transformer-based self-attention provides the same benefit automatically and more powerfully. The multi-head attention mechanism learns to attend over all token pairs simultaneously, implicitly capturing hierarchical syntactic and semantic relationships without any explicit tree structure. Models like BERT and Flan-T5 therefore learn compositional sentiment signals (negation, intensifiers, contrastive conjunctions) directly from data during pre-training on large corpora, replacing the need for hand-annotated parse trees entirely.

In [5]:
from datasets import load_dataset
raw = load_dataset('stanfordnlp/sst2')
raw

README.md:   0%|          | 0.00/5.27k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 1821
    })
})

In [6]:
raw['train'][5], raw['validation'][5]

({'idx': 5,
  'sentence': "that 's far too tragic to merit such superficial treatment ",
  'label': 0},
 {'idx': 5,
  'sentence': 'although laced with humor and a few fanciful touches , the film is a refreshingly serious look at young women . ',
  'label': 1})

In [7]:
# Select a reproducible random sample of 2000 examples from the training split
train_size = 2000
train_ds = raw['train'].shuffle(seed=42).select(range(train_size))
label_names = train_ds.features['label'].names
eval_ds  = raw['validation']


In [8]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
MODEL_NAME = 'google/flan-t5-small'
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
flan = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

## Zero-Shot and Few-Shot Prompting

One of the major benefits of today's generative models is that they can often be used effectively with no supervised, task-specific training (i.e. fine tuning). This avoids the time and expense needed to compile and train on a labeled dataset and is called zero-shot or few-shot prompting. However, using a generative model makes performance evaluation more complex, since the set of possible outputs is not pre-defined (i.e. the model can potentially produce any tokens in its vocabulary).

**TODO**:

- Define a function to format zero-shot prompts.
- Define a function to format few-shot prompts using the examples provided.
- Define a function that will normalize the generated output for evaluation.

**Q**: Discuss one additional benefit and drawback to using a generative model with zero-shot or few-shot prompting in place of traditional supervised learning methods.

**A**: **Benefit** -- Zero/few-shot prompting requires no labeled training data and no training loop. You can deploy a model on a brand-new task in minutes by simply describing the task in natural language, which is extremely valuable for low-resource domains or rapid prototyping. The same model can also handle many different tasks without retraining, unlike a task-specific classifier.

**Drawback** -- Output control is fundamentally harder. Because the generative model can produce any string, a post-processing step (such as norm_label here) is required to map free-text outputs back to a fixed label set, and the model may still produce outputs that do not match any expected class (e.g., 'The sentiment is mixed'). In contrast, a supervised classifier with a fixed output head always produces a valid class probability distribution, making evaluation and deployment cleaner and more reliable.

In [9]:
FEW_SHOTS = [
    ('This movie was fantastic and heartwarming.', 'positive'),
    ('The plot was boring and predictable.', 'negative'),
]

def zshot_prompt(text):
    """Wrap the input in a zero-shot classification prompt."""
    return (
        "Classify the sentiment of the following sentence as 'positive' or 'negative'.\n"
        f"Sentence: {text}\n"
        "Sentiment:"
    )

def fshot_prompt(text):
    """Wrap the input in a few-shot prompt that includes labeled examples."""
    header = "Classify the sentiment of each sentence as 'positive' or 'negative'.\n\n"
    examples = "".join(
        f"Sentence: {sent}\nSentiment: {label}\n\n"
        for sent, label in FEW_SHOTS
    )
    query = f"Sentence: {text}\nSentiment:"
    return header + examples + query

def norm_label(s: str):
    """Normalize model output text to 'positive' or 'negative'."""
    s = (s or '').lower().strip()
    # Match any string containing a positive keyword
    if any(w in s for w in ['positive', 'pos', '1', 'great', 'good', 'excellent']):
        return 'positive'
    # Match any string containing a negative keyword
    if any(w in s for w in ['negative', 'neg', '0', 'bad', 'poor', 'terrible']):
        return 'negative'
    return s.strip()  # return as-is; will count as wrong during evaluation

@torch.no_grad()  # tells pytorch not to store any gradients
def flan_predict(texts, mode='zero', max_new_tokens=3):
    preds = []
    for t in texts:
        prompt = zshot_prompt(t) if mode == 'zero' else fshot_prompt(t)
        inputs = tok(prompt, return_tensors='pt')
        outputs = flan.generate(**inputs, max_new_tokens=max_new_tokens)
        out = tok.batch_decode(outputs, skip_special_tokens=True)[0]
        preds.append(norm_label(out))
    return preds

print('Zero-shot:', flan_predict(['I loved this movie', 'This was terrible'], mode='zero'))
print('Few-shot :', flan_predict(['I loved this movie', 'This was terrible'], mode='few'))

Zero-shot: ['positive', 'negative']
Few-shot : ['positive', 'negative']


#### Evaluate Prompting Methods

**TODO**:

- Define a function (or use a pre-existing implementation) that computes accuracy, with lists of labels and predictions as input.
- Use your `flan_predict` function from above to produce zero-shot and few-shot predictions over your evaluation data.

**Q**: Reflect on the performance of these two methods. Was there anything that surprised you?

**A**: Flan-T5-Small typically achieves around 70-80% accuracy zero-shot on SST-2, which is surprisingly strong for a model that has never been shown a single labeled example from this task. The few-shot method generally improves accuracy by a few percentage points by giving the model concrete examples of the output format, which reduces cases where the model produces an ambiguous or malformed label. What is particularly surprising is how well zero-shot works at all given that Flan-T5-Small has only ~80M parameters -- much smaller than the GPT-class models usually associated with strong zero-shot ability. This reflects the effectiveness of instruction-tuning during Flan pre-training. On the other hand, the confusion matrices often reveal a class imbalance in errors: the model may systematically over-predict 'positive', suggesting that generation bias (rather than lack of language understanding) is the main failure mode at this scale.

In [10]:
def accuracy(y_true, y_pred):
    """Compute fraction of predictions that exactly match the true labels."""
    if not y_true:
        return 0.0
    correct = sum(t == p for t, p in zip(y_true, y_pred))
    return correct / len(y_true)

def label_to_str(y):  # the SST-2 dataset stores sentiment labels as integers
    return label_names[int(y)]

eval_texts  = [ex['sentence'] for ex in eval_ds]
eval_labels = [label_to_str(ex['label']) for ex in eval_ds]

# Run both prompting strategies over the full validation split
z_preds = flan_predict(eval_texts, mode='zero')
f_preds = flan_predict(eval_texts, mode='few')

z_acc = accuracy(eval_labels, z_preds)
f_acc = accuracy(eval_labels, f_preds)
print({'zero_shot_acc': round(z_acc, 4), 'few_shot_acc': round(f_acc, 4)})
print('Zero-Shot Confusion Matrix:\n', confusion_matrix(eval_labels, z_preds))
print('Few-Shot Confusion Matrix:\n',  confusion_matrix(eval_labels, f_preds))

{'zero_shot_acc': 0.8693, 'few_shot_acc': 0.8773}
Zero-Shot Confusion Matrix:
 [[380  48]
 [ 66 378]]
Few-Shot Confusion Matrix:
 [[381  47]
 [ 60 384]]


## Model Fine Tuning

Now we will fine tune the same Flan-T5 model on the SST-2 training dataset and compare performance. Since we are still using a generative model, the data needs to be formatted to support generation as we did above. We also need to define a custom metric function for the model to use during training, ensuring that the output matches our expected labels for evaluation.

**TODO**:

- Define a function to format data to use in a generative model training pipeline.
- Use your `norm_label` function from above to process the model output for evaluation.

**Q**: How would the model training pipeline change if we were using a representative model (i.e. an encoder-side transformer model like BERT) instead of a generative model? Which type of model makes more sense when doing a supervised training task?

**A**: With an encoder-only model like BERT, the pipeline would change substantially. We would use AutoModelForSequenceClassification instead of AutoModelForSeq2SeqLM, and the inputs would be tokenized directly without any prompt wrapping. The model would output a probability vector over a fixed set of class labels via a linear classification head added on top of the [CLS] token embedding, and training would use cross-entropy loss over those logits. Evaluation would simply take argmax of the output probabilities -- no norm_label post-processing would be needed because the model cannot produce out-of-vocabulary responses. DataCollatorForSeq2Seq and Seq2SeqTrainer would be replaced by DataCollatorWithPadding and the standard Trainer.

For a supervised classification task like SST-2, an encoder-only model makes more sense. It is more computationally efficient (no decoder stack), has a lower parameter count for equivalent representational capacity, and produces deterministic bounded outputs (class probabilities). The generative approach is valuable in zero/few-shot settings because the decoder can leverage language modeling priors to solve tasks without any labeled data, but once labeled training data is available the classification head of a BERT-style model is a more direct, efficient, and reliable fit for the task.

In [11]:
def format_data(example):
    """Wrap the raw sentence in the same generation-style prompt used for prompting,
    and convert the integer label to a human-readable string target."""
    inp = (
        "Classify the sentiment of the following sentence as 'positive' or 'negative'.\n"
        f"Sentence: {example['sentence']}\n"
        "Sentiment:"
    )
    tgt = label_to_str(example['label'])  # integer 0/1 -> 'negative'/'positive'
    return {'input_text': inp, 'target_text': tgt}

train_formatted = train_ds.map(format_data)  # use .map() in place of an explicit loop
eval_formatted  = eval_ds.map(format_data)

def tokenize_fn(batch):
    model_inputs = tok(batch['input_text'], truncation=True)  # convert text to token ids
    labels = tok(batch['target_text'], truncation=True)        # tokenize the target strings
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

train_toks = train_formatted.map(tokenize_fn, batched=True, remove_columns=train_formatted.column_names)
eval_toks  = eval_formatted.map(tokenize_fn,  batched=True, remove_columns=eval_formatted.column_names)

data_collator = DataCollatorForSeq2Seq(tok, model=flan)  # handles dynamic padding during batching


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

In [12]:
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    pred_texts = tok.batch_decode(preds, skip_special_tokens=True)  # token ids -> text
    labels_clean = []
    for row in labels:
        row = [id for id in row if id != -100]  # remove padding tokens
        labels_clean.append(row)
    ref_texts = tok.batch_decode(labels_clean, skip_special_tokens=True)

    # Normalize generated outputs so minor formatting differences do not penalise accuracy
    preds_norm = [norm_label(p) for p in pred_texts]
    return {'accuracy': accuracy(ref_texts, preds_norm)}


In [13]:
training_args = Seq2SeqTrainingArguments(
    output_dir='outputs/flan_t5_sst2',
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='no',
    predict_with_generate=True,
    logging_steps=50,
    report_to=[],
)

trainer = Seq2SeqTrainer(
    model=flan,
    args=training_args,
    train_dataset=train_toks,
    eval_dataset=eval_toks,
    processing_class=tok,  # renamed from `tokenizer` in transformers 5.x
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train();

Epoch,Training Loss,Validation Loss,Accuracy
1,0.151311,0.156988,0.883028
2,0.167840,0.151256,0.886468


In [14]:
eval_out = trainer.predict(eval_toks)
ft_acc = round(eval_out.metrics["test_accuracy"], 4)
print({'finetuned_flan_t5_eval_accuracy': ft_acc})
# Decode predicted sequences
pred_texts = tok.batch_decode(eval_out.predictions, skip_special_tokens=True)

# Decode reference (true) sequences, removing padding (-100)
labels_clean = [[id for id in row if id != -100] for row in eval_out.label_ids]
ref_texts = tok.batch_decode(labels_clean, skip_special_tokens=True)
print(confusion_matrix(ref_texts, pred_texts, labels=["positive", "negative"]))

{'finetuned_flan_t5_eval_accuracy': 0.8865}
[[398  46]
 [ 53 375]]


### Model Comparison

**Q**: Reflect on the performance of your three methods. Is there anything that was surprising? Would you do anything to improve the performance of any of the methods? Are there any other methods that you would like to compare?

**A**: 
The three methods — zero-shot, few-shot, and fine-tuning — achieved accuracies of 86.93%, 87.73%, and 88.65% respectively. The most surprising finding was how small the gap between all three methods turned out to be. Typically, fine-tuning is expected to outperform zero-shot prompting by 10–15 percentage points, but here the margin was only ~1.7%. This suggests that Flan-T5-Small's instruction-tuning during pre-training already equipped it with strong sentiment understanding, leaving little room for improvement on a binary task like SST-2 — a phenomenon known as the ceiling effect.
It was also surprising that zero-shot alone achieved 86.93%, which is competitive with much larger models. This highlights the effectiveness of instruction-tuning as a pre-training strategy, even at small model scales.
To improve performance, I would consider the following:

**Zero/few-shot:** Increase the number of few-shot examples from 2 to 8–16, or apply constrained decoding (restricting beam search to only generate 'positive' or 'negative') to eliminate any out-of-label outputs that get penalized during evaluation.

**Fine-tuning:** Train on the full SST-2 training set (~67K examples) instead of just 2,000, which would likely push accuracy toward 91–92%. Alternatively, replace the seq2seq generation head with a linear classification head on the encoder output, making the model a true discriminative classifier rather than a generative one — this is more efficient and better suited to fixed-label tasks.

**All methods:** Experiment with a larger Flan-T5 variant (base or large) to see how much of the performance is attributable to model scale versus training strategy.

**Other methods I would like to compare include:**

RoBERTa or BERT fine-tuned with a classification head — the standard encoder-side baseline for SST-2, which typically achieves ~93% on the full dataset and would serve as a strong upper-bound comparison.
Parameter-efficient fine-tuning (LoRA or Prefix-Tuning) — updates fewer than 1% of model parameters while achieving near full fine-tuning accuracy, making it a practical middle ground between prompting and full fine-tuning.
GPT-3.5 or GPT-4 zero-shot — to quantify how much of the zero-shot gap is simply a function of model scale, since larger models tend to be significantly better at following zero-shot instructions on classification tasks.

Overall, the results reinforce that SST-2 is a relatively saturated benchmark for modern pre-trained models, and a more complex task such as multi-class emotion detection or domain-specific sentiment would likely reveal much larger and more meaningful differences between these three approaches.

In [15]:
summary = {
    'zero_shot_flan_t5_acc': round(z_acc, 4),
    'few_shot_flan_t5_acc': round(f_acc, 4),
    'finetuned_flan_t5_acc' : ft_acc,
}
summary

{'zero_shot_flan_t5_acc': 0.8693,
 'few_shot_flan_t5_acc': 0.8773,
 'finetuned_flan_t5_acc': 0.8865}